# Lichess Game Analysis — Data Feasibility

This notebook is a first-pass feasibility test for a possible chess analysis project using a large Lichess dataset.

The goal is not to build the final analysis yet. I’m trying to answer a simpler question first:

> Can this large, wide chess dataset be reduced into a clean game-level table that I understand well enough to analyze?

I’m intentionally starting with the columns I can explain: game metadata, player ratings, result, termination type, time control, category, weekday, and later selected evaluation/clock columns if they prove useful.


## 1. Setup

The project is organized with separate raw and processed data folders. I’m also preparing a SQLite database path early because, if this dataset passes the feasibility test, the main analysis should move into SQL instead of staying fully in Pandas.


In [43]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DB_PATH = PROCESSED_DIR / "lichess.db"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


## 2. Load selected columns only

The raw dataset has hundreds of repeated move, evaluation, and clock columns. I’m not loading everything yet.

For this first pass, I’m keeping:
- core game metadata,
- player ratings and rating changes,
- result and termination information,
- opening information,
- time-control information,
- the first 80 evaluation columns,
- the first 80 clock columns.

I’m dropping the move notation columns for now because analyzing move strings would require more chess-specific knowledge and would distract from the main data analysis goal.


In [44]:
CSV_PATH = RAW_DIR / "lichess_200k.csv"

metadata_cols = [
    "Index",
    "Black", "BlackElo", "BlackRatingDiff",
    "White", "WhiteElo", "WhiteRatingDiff",
    "Date", "UTCDate", "UTCTime",
    "ECO", "Opening", "Result",
    "Termination", "TimeControl",
    "Category", "Weekday"
]

eval_cols = [f"Eval_ply_{i}" for i in range(1, 81)]
clock_cols = [f"Clock_ply_{i}" for i in range(1, 81)]

selected_cols = metadata_cols + eval_cols + clock_cols


## 3. Load a 20k-row sample

I’m starting with a sample instead of the full CSV. The point is to understand whether the structure is usable before committing to the full dataset.

If this sample behaves well, I can later repeat the same cleaning logic on a larger subset or the full selected-column dataset.


In [45]:
df_sample = pd.read_csv(
    CSV_PATH,
    usecols=selected_cols,
    nrows=20_000,
    low_memory=False
)

df_sample.shape


(20000, 177)

In [46]:
df_sample.head()


,Index,Black,BlackElo,BlackRatingDiff,Date,ECO,Opening,Result,Termination,TimeControl,UTCDate,UTCTime,White,WhiteElo,WhiteRatingDiff,Eval_ply_1,Eval_ply_2,Eval_ply_3,Eval_ply_4,Eval_ply_5,Eval_ply_6,Eval_ply_7,Eval_ply_8,Eval_ply_9,Eval_ply_10,Eval_ply_11,Eval_ply_12,Eval_ply_13,Eval_ply_14,Eval_ply_15,Eval_ply_16,Eval_ply_17,Eval_ply_18,Eval_ply_19,Eval_ply_20,Eval_ply_21,Eval_ply_22,Eval_ply_23,Eval_ply_24,Eval_ply_25,Eval_ply_26,Eval_ply_27,Eval_ply_28,Eval_ply_29,Eval_ply_30,Eval_ply_31,Eval_ply_32,Eval_ply_33,Eval_ply_34,Eval_ply_35,Eval_ply_36,Eval_ply_37,Eval_ply_38,Eval_ply_39,Eval_ply_40,Eval_ply_41,Eval_ply_42,Eval_ply_43,Eval_ply_44,Eval_ply_45,Eval_ply_46,Eval_ply_47,Eval_ply_48,Eval_ply_49,Eval_ply_50,Eval_ply_51,Eval_ply_52,Eval_ply_53,Eval_ply_54,Eval_ply_55,Eval_ply_56,Eval_ply_57,Eval_ply_58,Eval_ply_59,Eval_ply_60,Eval_ply_61,Eval_ply_62,Eval_ply_63,Eval_ply_64,Eval_ply_65,Eval_ply_66,Eval_ply_67,Eval_ply_68,Eval_ply_69,Eval_ply_70,Eval_ply_71,Eval_ply_72,Eval_ply_73,Eval_ply_74,Eval_ply_75,Eval_ply_76,Eval_ply_77,Eval_ply_78,Eval_ply_79,Eval_ply_80,Clock_ply_1,Clock_ply_2,Clock_ply_3,Clock_ply_4,Clock_ply_5,Clock_ply_6,Clock_ply_7,Clock_ply_8,Clock_ply_9,Clock_ply_10,Clock_ply_11,Clock_ply_12,Clock_ply_13,Clock_ply_14,Clock_ply_15,Clock_ply_16,Clock_ply_17,Clock_ply_18,Clock_ply_19,Clock_ply_20,Clock_ply_21,Clock_ply_22,Clock_ply_23,Clock_ply_24,Clock_ply_25,Clock_ply_26,Clock_ply_27,Clock_ply_28,Clock_ply_29,Clock_ply_30,Clock_ply_31,Clock_ply_32,Clock_ply_33,Clock_ply_34,Clock_ply_35,Clock_ply_36,Clock_ply_37,Clock_ply_38,Clock_ply_39,Clock_ply_40,Clock_ply_41,Clock_ply_42,Clock_ply_43,Clock_ply_44,Clock_ply_45,Clock_ply_46,Clock_ply_47,Clock_ply_48,Clock_ply_49,Clock_ply_50,Clock_ply_51,Clock_ply_52,Clock_ply_53,Clock_ply_54,Clock_ply_55,Clock_ply_56,Clock_ply_57,Clock_ply_58,Clock_ply_59,Clock_ply_60,Clock_ply_61,Clock_ply_62,Clock_ply_63,Clock_ply_64,Clock_ply_65,Clock_ply_66,Clock_ply_67,Clock_ply_68,Clock_ply_69,Clock_ply_70,Clock_ply_71,Clock_ply_72,Clock_ply_73,Clock_ply_74,Clock_ply_75,Clock_ply_76,Clock_ply_77,Clock_ply_78,Clock_ply_79,Clock_ply_80,Category,Weekday
0,0,albertoPlasta,906,13.00,2019.04.30,B15,Caro-Kann Defense,0-1,Normal,300+3,2019.04.30,22:00:24,KingMateo,971,-12.00,0.25,0.15,-0.15,0.47,0.13,0.72,0.66,0.51,0.0,1.63,0.56,0.79,-0.65,-1.01,-1.44,-1.12,-1.69,-1.58,-2.87,-2.08,-3.4,-3.27,-3.56,-2.32,-10.81,-10.83,-11.13,-11.02,-10.65,-10.14,-10.67,-10.53,-11.15,-10.86,-10.99,-9.95,-11.51,-10.24,-10.34,-10.21,-10.41,-9.05,-10.9,-10.61,-10.27,-9.57,-9.9,-8.16,-8.09,-7.88,-7.95,-7.56,#-15,-70.52,-69.37,-22.66,#-10,-13.15,-22.33,-22.55,-21.08,-18.77,-66.25,-25.68,-20.97,-19.73,#-19,-65.66,#-5,-14.39,#-5,#-4,#-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0:05:00,0:05:00,0:04:55,0:04:57,0:04:46,0:04:49,0:04:42,0:04:49,0:04:35,0:04:43,0:04:26,0:04:40,0:04:20,0:04:34,0:04:17,0:04:22,0:03:50,0:04:03,0:03:39,0:03:57,0:03:33,0:03:35,0:03:29,0:03:21,0:03:18,0:03:16,0:03:18,0:03:18,0:03:19,0:03:16,0:03:12,0:03:01,0:03:08,0:03:01,0:03:03,0:02:47,0:02:53,0:02:44,0:02:48,0:02:32,0:02:43,0:02:15,0:02:30,0:02:01,0:02:25,0:01:49,0:02:22,0:01:48,0:02:15,0:01:47,0:02:11,0:01:47,0:01:58,0:01:41,0:01:57,0:01:39,0:01:55,0:01:37,0:01:56,0:01:27,0:01:50,0:01:19,0:01:36,0:01:11,0:01:29,0:00:56,0:01:24,0:00:35,0:01:13,0:00:37,0:01:11,0:00:32,0:01:09,0:00:27,NaN,NaN,NaN,NaN,NaN,NaN,Blitz,Tuesday
1,1,Luckystriker,1296,28.00,2019.04.30,C50,Italian Game,0-1,Normal,300+0,2019.04.30,22:00:13,G_i_n,1312,-10.00,0.12,0.37,0.23,0.15,0.21,0.46,0.36,0.38,0.48,0.36,0.38,0.58,0.54,0.32,0.6,1.25,0.66,0.68,0.3,0.34,0.37,0.3,-0.21,-0.02,-0.41,0.17,0.23,0.43,0.55,1.52,1.42,1.95,0.0,2.13,0.09,0.11,0.0,0.0,0.0,-0.29,-2.0,-1.9,-4.47,-4.29,-11.0,-10.71,-9.83,-7.03,-7.45,-3.05,-60.79,-7.73,-13.11,-2.36,#-12,#-11,#-11,#-10,#-6,#-10,#-10,-7.39,#-7,#-6,#-2,#-1,#-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0:05:00,0:05:00,0:04:59,0:04:59,0:04:59,0:04:58,0:04:58,0:04:55,0:04:53,0:04:50,0:04:51,0:04:42,0:04:43,0:04:42,0:04:37,0:04:24,0:04:35,0:04:

## 4. Initial game-level structure

Before touching the repeated evaluation and clock columns, I first inspect the game-level metadata. These are the columns that define the basic row grain and analysis context, excluding Eval cols and Clock cols.


In [47]:
df_sample[
    [
        "Index", "Date", "UTCDate", "UTCTime",
        "Result", "Termination", "TimeControl",
        "Category", "Weekday", "ECO", "Opening"
    ]
].head()


,Index,Date,UTCDate,UTCTime,Result,Termination,TimeControl,Category,Weekday,ECO,Opening
0,0,2019.04.30,2019.04.30,22:00:24,0-1,Normal,300+3,Blitz,Tuesday,B15,Caro-Kann Defense
1,1,2019.04.30,2019.04.30,22:00:13,0-1,Normal,300+0,Blitz,Tuesday,C50,Italian Game
2,2,2019.04.30,2019.04.30,22:00:41,1-0,Normal,600+0,Rapid,Tuesday,C41,Philidor Defense #2
3,3,2019.04.30,2019.04.30,22:00:43,0-1,Normal,60+0,Bullet,Tuesday,B06,Modern Defense
4,4,2019.04.30,2019.04.30,22:00:46,1-0,Normal,180+0,Blitz,Tuesday,B32,Sicilian Defense: Loewenthal Variation


## 5. Row identifiers and date columns

The raw CSV contains index-like columns that appear to come from earlier exports. I do not want to treat them as meaningful chess variables unless they prove useful.

I also check whether `Date` and `UTCDate` contain the same values. If they are identical in the sample, keeping both would be redundant.


In [48]:
df_sample["Index"].is_unique


True

In [49]:
df_sample[["Date", "UTCDate"]].head(10)


,Date,UTCDate
0,2019.04.30,2019.04.30
1,2019.04.30,2019.04.30
2,2019.04.30,2019.04.30
3,2019.04.30,2019.04.30
4,2019.04.30,2019.04.30
5,2019.04.30,2019.04.30
6,2019.04.30,2019.04.30
7,2019.04.30,2019.04.30
8,2019.04.30,2019.04.30
9,2019.04.30,2019.04.30


In [50]:
(df_sample["Date"] == df_sample["UTCDate"]).mean()


np.float64(1.0)

### Cleaning decision

`Date` and `UTCDate` are identical in this sample, so I keep `Date` and drop `UTCDate`.

The raw dataset also contains duplicate index-like columns. They look like export artifacts rather than meaningful game identifiers, so I create a clean `game_id` based on the current row order for this feasibility test.


In [51]:
# Drop redundant/date and export-artifact columns
df_sample = df_sample.drop(columns=["UTCDate", "Index", "Index.1"], errors="ignore")

# Create a clean row-based game ID
df_sample = df_sample.reset_index(drop=True)
df_sample["game_id"] = df_sample.index + 1


## 6. Data Quality Checks

Before moving into analysis, I need to check whether the main game-level columns are reliable enough to use. This dataset contains hundreds of move, evaluation, and clock columns, so not all missing values mean the same thing.

For example, missing values in later move/evaluation/clock columns are expected because chess games do not all have the same length. A game that ends after 40 plies will naturally have missing values for later ply columns. However, missing values in columns such as result, rating, time control, category, or termination would directly affect the analysis.

In this section, I focus on the most important game-level quality questions:

- Are there missing values in key metadata columns?
- Are there duplicated game records?
- Are the main categorical variables consistent?
- Are the rating columns usable as numeric variables?
- Does the dataset still appear to follow one row per game?

These checks are important because the early insights in this notebook depend on the dataset being structurally reliable. If the game-level columns are clean enough, then the project can safely move into SQL-based analysis in the next notebook.

In [57]:
key_metadata_cols = [
    "WhiteElo",
    "BlackElo",
    "WhiteRatingDiff",
    "BlackRatingDiff",
    "Date",
    "UTCTime",
    "ECO",
    "Opening",
    "Result",
    "Termination",
    "TimeControl",
    "Category",
    "Weekday"
]

df_sample[key_metadata_cols].isna().sum().sort_values(ascending=False)

WhiteRatingDiff    42
BlackRatingDiff    42
WhiteElo            0
BlackElo            0
Date                0
UTCTime             0
ECO                 0
Opening             0
Result              0
Termination         0
TimeControl         0
Category            0
Weekday             0
dtype: int64

### 6.1 Missing Values in Key Metadata Columns

I first checked missing values in the main game-level metadata columns rather than all columns in the dataset. This is because the dataset contains many move, evaluation, and clock columns, and missing values in later ply columns are expected when games end before reaching those moves.

The key metadata columns are mostly complete. Columns such as `WhiteElo`, `BlackElo`, `Date`, `UTCTime`, `ECO`, `Opening`, `Result`, `Termination`, `TimeControl`, `Category`, and `Weekday` do not contain missing values in this sample.

The only missing values appear in `WhiteRatingDiff` and `BlackRatingDiff`, with 42 missing values each. Since these columns describe the rating change after the game, the missing values may be related to specific game conditions such as draws, unrated games, or special termination cases. I will inspect these rows separately before deciding whether to drop, keep, or ignore these columns in later analysis.


In [58]:
df_sample.loc[
    df_sample["WhiteRatingDiff"].isna() | df_sample["BlackRatingDiff"].isna(),
    ["Result", "WhiteElo", "BlackElo", "WhiteRatingDiff", "BlackRatingDiff"]
]["Result"].value_counts(dropna=False)

df_sample.loc[
    df_sample["WhiteRatingDiff"].isna() | df_sample["BlackRatingDiff"].isna(),
    ["Result", "Termination", "Category", "WhiteElo", "BlackElo", "WhiteRatingDiff", "BlackRatingDiff"]
].head(20)

,Result,Termination,Category,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff
389,1-0,Normal,Blitz,1943,1948,NaN,NaN
677,0-1,Normal,Blitz,1741,1656,NaN,NaN
1718,0-1,Normal,Rapid,1792,2327,NaN,NaN
3217,1-0,Normal,Blitz,1777,1723,NaN,NaN
3883,1-0,Normal,Rapid,1671,1919,NaN,NaN
5120,1-0,Time forfeit,Blitz,1527,1500,NaN,NaN
5751,1-0,Normal,Blitz,1875,1877,NaN,NaN
5916,0-1,Normal,Classical,1934,1689,NaN,NaN
5988,0-1,Normal,Rapid,1683,1914,NaN,NaN
6860,0-1,Normal,Rapid,1086,2023,NaN,NaN


In [62]:
missing_rating_diff_count = df_sample[
    df_sample["WhiteRatingDiff"].isna() | df_sample["BlackRatingDiff"].isna()
].shape[0]

missing_rating_diff_pct = missing_rating_diff_count / len(df_sample) * 100

missing_rating_diff_count, round(missing_rating_diff_pct, 2)

(42, 0.21)

`WhiteRatingDiff` and `BlackRatingDiff` each have 42 missing values, which represents about 0.21% of the 20,000-row sample. The missing values occur together in both columns. After inspecting the affected rows, they do not appear to be explained by draws or unusual termination types, since many of these games ended normally and had clear winners.

Because the affected share is very small, I will keep these rows for general game-level analysis. However, I will avoid using `WhiteRatingDiff` and `BlackRatingDiff` directly unless the missing values are filtered or handled explicitly. I will not fill these values with zero, because a missing rating change is not the same as a rating change of zero.

### 6.2 Duplicate Records Check

After checking missing values, the next step is to check whether the dataset contains duplicated game records. Duplicate rows could distort the analysis by making certain games, players, openings, or outcomes appear more common than they actually are.

I will start with an exact duplicate check across the full dataframe. This checks whether any rows are repeated exactly.

Then, I will do a second duplicate check using a smaller set of game-identifying columns, such as the players, date, time, result, and time control. This is useful because the dataset does not contain a clean original game ID, and the raw index columns appear to be export artifacts rather than reliable identifiers.

The goal is not to prove with absolute certainty that every game is unique, but to check whether there are obvious repeated records that could affect the analysis.


In [66]:
exact_duplicates = df_sample.duplicated().sum()
exact_duplicates

np.int64(0)

In [ ]:
duplicate_check_cols = [
    "White",
    "Black",
    "Date",
    "UTCTime",
    "Result",
    "TimeControl"
]

df_sample.loc[
    df_sample.duplicated(subset=duplicate_check_cols, keep=False),
    duplicate_check_cols + ["Opening", "Termination", "Category"]
].sort_values(["Date", "UTCTime", "White", "Black"])

,White,Black,Date,UTCTime,Result,TimeControl,Opening,Termination,Category


### Duplicate Check Observation
The duplicate checks did not find any duplicated rows. There were no exact duplicates across the full dataframe, and there were also no duplicated records when using the selected game-identifying metadata columns.

This supports the assumption that each row in the sample represents a distinct chess game. Since there are no obvious repeated game records, I do not need to remove duplicates before continuing with the analysis.


## 6.3 Categorical overview

Now I inspect the main categorical columns:

- `Result`: who won the game,
- `Termination`: how the game ended,
- `Category`: Bullet, Blitz, Rapid, or Classical,
- `TimeControl`: raw time format,
- `Weekday`: day of play.

This tells me whether the dataset has enough variation to support meaningful analysis.


In [52]:
for col in ["Result", "Termination", "Category", "TimeControl", "Weekday"]:
    print(f"\n{col}".upper())
    print(df_sample[col].value_counts(dropna=False).head(15))



RESULT
Result
1-0        10118
0-1         9409
1/2-1/2      473
Name: count, dtype: int64

TERMINATION
Termination
Normal              15288
Time forfeit         4708
Rules infraction        4
Name: count, dtype: int64

CATEGORY
Category
Blitz        8630
Rapid        4649
Bullet       4213
Classical    2508
Name: count, dtype: int64

TIMECONTROL
TimeControl
600+0      3673
60+0       2896
300+0      2802
180+0      2216
900+15     2202
300+3      1804
180+2      1503
120+1       910
900+0       128
15+0        122
300+5        89
30+0         86
1200+10      81
300+8        79
600+5        71
Name: count, dtype: int64

WEEKDAY
Weekday
Thursday     6366
Friday       6198
Saturday     4100
Wednesday    2835
Tuesday       501
Name: count, dtype: int64


### First observations

The game-level columns look usable. `Result`, `Termination`, `Category`, and `TimeControl` all contain interpretable values.

One early pattern is that Bullet games should be watched closely because faster time controls may change *how* games end, especially through time forfeits.


### 6.4 Numeric and Date Sanity Checks

After checking missing values and duplicate records, I want to make sure that the main numeric and date columns are reasonable. Even if a column has no missing values, it can still contain unrealistic values that would affect the analysis.

For this dataset, the most important numeric columns are the player ratings and rating differences. I'll check whether `WhiteElo`, `BlackElo`, `WhiteRatingDiff`, and `BlackRatingDiff` have reasonable ranges and do not contain impossible or suspicious values.

I'll also inspect the date range of the sample. This is important because the weekday distribution was uneven, with some days appearing much more often than others. Checking the minimum and maximum dates helps determine whether this is likely caused by the dataset collection window rather than actual chess-playing behavior.


In [67]:
numeric_cols = [
    "WhiteElo",
    "BlackElo",
    "WhiteRatingDiff",
    "BlackRatingDiff"
]

df_sample[numeric_cols].describe()

,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff
count,"20,000.00","20,000.00","19,958.00","19,958.00"
mean,"1,511.15","1,511.49",4.18,3.11
std,320.82,320.34,30.80,30.12
min,800.00,800.00,-455.00,-507.00
25%,"1,279.00","1,282.00",-10.00,-10.00
50%,"1,500.00","1,501.00",3.00,-3.00
75%,"1,733.00","1,731.00",11.00,11.00
max,"2,869.00","2,763.00",456.00,543.00


### Numeric Columns Observation

The player rating columns appear reasonable overall. `WhiteElo` and `BlackElo` range from 800 to the high 2000s, which is plausible for online chess ratings.

However, the rating-difference columns have a few large values. `WhiteRatingDiff` reaches a maximum of 456, while `BlackRatingDiff` reaches a maximum of 543. These values are much higher than the average rating changes, so they are worth inspecting before deciding whether the columns are reliable.

I will not treat these values as errors automatically. Large rating changes can happen in online rating systems, especially for newer or provisional players. Instead, I will inspect the rows with the largest absolute rating changes and check whether the surrounding game information still looks valid, this sounds like the most logical step to do at this stage.


In [70]:
rating_diff_outliers = df_sample.assign(
    abs_white_rating_diff=df_sample["WhiteRatingDiff"].abs(),
    abs_black_rating_diff=df_sample["BlackRatingDiff"].abs()
)

rating_diff_outliers.sort_values(
    ["abs_white_rating_diff", "abs_black_rating_diff"],
    ascending=False
)[
    [
        "WhiteElo",
        "BlackElo",
        "WhiteRatingDiff",
        "BlackRatingDiff",
        "Result",
        "Termination",
        "Category",
        "TimeControl"
    ]
].head(20)

,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,Result,Termination,Category,TimeControl
19770,1166,1919,456.00,-20.00,1-0,Normal,Blitz,180+0
12677,1500,1093,-455.00,39.00,0-1,Normal,Rapid,420+9
13118,1500,2165,404.00,-377.00,1-0,Normal,Blitz,180+0
19985,1457,1858,393.00,-18.00,1-0,Normal,Blitz,300+2
591,1500,1200,-377.00,27.00,0-1,Normal,Blitz,240+3
13287,1500,1775,373.00,-10.00,1-0,Normal,Blitz,300+0
8280,1500,1759,359.00,-10.00,1-0,Normal,Blitz,300+0
10840,1500,1754,355.00,-11.00,1-0,Time forfeit,Blitz,180+0
11795,1500,1762,352.00,-15.00,1-0,Time forfeit,Bullet,15+0
11551,1500,1747,352.00,-12.00,1-0,Normal,Rapid,420+7


### Rating Difference Outlier Observation

After inspecting the rows with the largest absolute rating changes, the extreme values do not appear to be corrupted records. The games still have valid ratings, results, termination types, categories, and time controls.

The large rating changes seem to occur mostly when a lower-rated player beats a much higher-rated opponent, or when a player has a rating around 1500, which might indicate a newer or less stable rating. This makes the extreme values plausible in an online rating system rather than automatic errors.

For this project, I will keep these rows in the dataset. However, I will be cautious when using `WhiteRatingDiff` and `BlackRatingDiff` directly, because post-game rating changes may be affected by rating-system behavior such as provisional ratings. For most of the analysis, starting ratings like `WhiteElo`, `BlackElo`, and the rating difference between players will be more reliable.


### 6.5 Data Quality Summary

Overall, the game-level metadata appears reliable enough to continue the project.

The key metadata columns are mostly complete, with only a small number of missing values in `WhiteRatingDiff` and `BlackRatingDiff`. These missing values affect about 0.21% of the sample and do not impact the main game-level analysis.

The duplicate checks did not find exact duplicate rows or duplicated game-like records based on the selected metadata columns. This supports the assumption that each row represents a distinct chess game.

The player rating columns also appear reasonable overall. Some post-game rating changes are large, but after inspecting the affected rows, they seem plausible in an online rating system rather than corrupted records.

Based on these checks, I do not need to remove rows from the dataset at this stage. The main cleaning decisions are to drop the raw index artifact columns, keep the small number of missing rating-difference rows for general analysis, and use post-game rating-difference columns cautiously.


## 7. Create a winner column

The raw chess result format is compact:

- `1-0` means White won,
- `0-1` means Black won,
- `1/2-1/2` means Draw.

Creating a readable `winner` column will make the later SQL analysis easier.


In [53]:
df_sample["winner"] = df_sample["Result"].map({
    "1-0": "White",
    "0-1": "Black",
    "1/2-1/2": "Draw"
})

df_sample["winner"].value_counts(normalize=True).round(3)


winner
White   0.51
Black   0.47
Draw    0.02
Name: proportion, dtype: float64

## 8. Crosstab checks

A crosstab compares two categorical columns.

Here, I used it to ask questions like:

> Within each chess category, what proportion of games ended normally vs by time forfeit?

Using `normalize="index"` makes each row add up to 1, so the output is interpreted as row percentages.


In [54]:
termination_by_category = pd.crosstab(
    df_sample["Category"],
    df_sample["Termination"],
    normalize="index"
).round(3)

termination_by_category


Termination,Normal,Rules infraction,Time forfeit
Category,,,
Blitz,0.79,0.00,0.21
Bullet,0.51,0.00,0.49
Classical,0.91,0.00,0.09
Rapid,0.87,0.00,0.13


### Interpretation checkpoint

This table is important because it checks whether faster formats change how games end.

If Bullet has a much higher time-forfeit rate than Blitz, Rapid, or Classical, that might mean the clock is a major part of the story. This is a useful direction because it is easy to understand even without deep chess theory.

Small note: Since I'm not highly experienced in chess, I'm being careful not to overinterpret chess-specific variables too early. For now, I'm just focusing on patterns that are understandable from the data itself, such as time control, game result, termination type, and rating difference.


In [55]:
termination_by_result = pd.crosstab(
    df_sample["Termination"],
    df_sample["winner"],
    normalize="index"
).round(3)

termination_by_result


winner,Black,Draw,White
Termination,,,
Normal,0.47,0.03,0.50
Rules infraction,0.50,0.00,0.50
Time forfeit,0.48,0.01,0.51


### Interpretation checkpoint

This checks whether time forfeits are strongly biased toward White or Black.

If time-forfeit wins are close to evenly split between White and Black, then the key pattern is not color advantage. The better insight would be:

> Faster games are more likely to end by time forfeit, but time-forfeit wins do not appear strongly to be color-biased in this sample.


In [56]:
winner_by_category_and_termination = pd.crosstab(
    [df_sample["Category"], df_sample["Termination"]],
    df_sample["winner"],
    normalize="index"
).round(3)

winner_by_category_and_termination


winner                      Black  Draw  White
Category  Termination                         
Blitz     Normal             0.46  0.03   0.51
          Time forfeit       0.48  0.01   0.52
Bullet    Normal             0.47  0.02   0.50
          Time forfeit       0.49  0.00   0.51
Classical Normal             0.48  0.04   0.48
          Rules infraction   0.00  0.00   1.00
          Time forfeit       0.42  0.00   0.58
Rapid     Normal             0.46  0.03   0.51
          Rules infraction   1.00  0.00   0.00
          Time forfeit       0.50  0.01   0.49

## 9. Feasibility decision so far

This dataset is intimidating because it is very wide, but the game-level metadata is usable.

My current decision:
- The raw move columns can stay out of scope for now.
- The metadata columns are strong enough for a game-level analysis.
- The early termination/category patterns are interesting.
- The project should move toward SQL after this feasibility notebook.
- Evaluation and clock columns can be added later only if they lead to understandable engineered features.

Next step is definitely creating a clean game-level table and export it to SQLite for SQL analysis.


## 10. Next analysis direction

The next layer should investigate rating advantage.

Questions to test in SQL:
- Does the higher-rated player usually win?
- Are upsets more common in Bullet or Blitz than in Rapid/Classical?
- Does time forfeit reduce the advantage of the higher-rated player?
- Are certain openings associated with more upsets, or are those patterns mostly explained by rating differences?

This keeps the project focused on explainable patterns instead of drowning in 600+ raw columns.
